# Notebook 2: Detecção de Pessoas
Neste notebook, vamos criar um detector que localiza pessoas em imagens usando o COCO dataset.

In [ ]:
import os
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

from core.dataset import COCOPersonDataset

In [ ]:
# Configurações
BATCH_SIZE = 4  # Menor devido à memória
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
class PersonDetectionDataset(torch.utils.data.Dataset):
    def __init__(self, coco_data, transform=None):
        self.coco = coco_data['coco']
        self.image_dir = coco_data['image_dir']
        self.image_ids = coco_data['image_ids']
        self.transform = transform
        
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        # Carregar imagem
        img_id = self.image_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        image = Image.open(os.path.join(self.image_dir, img_info['file_name']))
        
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        # Obter anotações
        ann_ids = self.coco.getAnnIds(imgIds=img_id, catIds=[1])  # catId 1 = person
        anns = self.coco.loadAnns(ann_ids)
        
        # Preparar boxes e labels
        boxes = []
        for ann in anns:
            bbox = ann['bbox']  # [x, y, width, height]
            # Converter para [x1, y1, x2, y2]
            boxes.append([
                bbox[0],
                bbox[1],
                bbox[0] + bbox[2],
                bbox[1] + bbox[3]
            ])
        
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.ones((len(boxes),), dtype=torch.int64)  # todas são pessoas
        
        image_id = torch.tensor([img_id])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)
        
        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': image_id,
            'area': area,
            'iscrowd': iscrowd
        }
        
        if self.transform is not None:
            image = self.transform(image)
        
        return image, target

In [ ]:
def get_model():
    # Carregar modelo pré-treinado
    model = fasterrcnn_resnet50_fpn(pretrained=True)
    
    # Modificar para detectar apenas pessoas (background + person = 2 classes)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
    
    return model

In [ ]:
def train_one_epoch(model, optimizer, data_loader, device):
    model.train()
    total_loss = 0
    
    for images, targets in data_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_loss += losses.item()
    
    return total_loss / len(data_loader)

In [ ]:
def evaluate(model, data_loader, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for images, targets in data_loader:
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            total_loss += losses.item()
    
    return total_loss / len(data_loader)

In [ ]:
def visualize_detections(image, predictions, threshold=0.5):
    # Converter tensor para numpy
    image = image.permute(1, 2, 0).cpu().numpy()
    image = (image * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406])
    image = np.clip(image, 0, 1)
    
    # Criar figura
    fig, ax = plt.subplots(1)
    ax.imshow(image)
    
    # Desenhar boxes
    boxes = predictions['boxes'].cpu().numpy()
    scores = predictions['scores'].cpu().numpy()
    
    for box, score in zip(boxes, scores):
        if score > threshold:
            x1, y1, x2, y2 = box
            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2,
                edgecolor='r',
                facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1, f'{score:.2f}', color='white', 
                   bbox=dict(facecolor='red', alpha=0.5))
    
    plt.axis('off')
    plt.show()

In [ ]:
# Preparar dataset
dataset = COCOPersonDataset()
person_data = dataset.get_person_data()

# Transform
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Criar datasets
train_dataset = PersonDetectionDataset(person_data, transform=transform)

# Split train/val
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

# Criar dataloaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=lambda x: tuple(zip(*x))
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE,
    collate_fn=lambda x: tuple(zip(*x))
)

In [ ]:
# Inicializar modelo
model = get_model().to(DEVICE)
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9)

# Loop de treino
train_losses = []
val_losses = []

for epoch in range(NUM_EPOCHS):
    # Treino
    train_loss = train_one_epoch(model, optimizer, train_loader, DEVICE)
    train_losses.append(train_loss)
    
    # Validação
    val_loss = evaluate(model, val_loader, DEVICE)
    val_losses.append(val_loss)
    
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}]')
    print(f'Train Loss: {train_loss:.4f}')
    print(f'Val Loss: {val_loss:.4f}')
    print('-' * 50)

In [ ]:
# Plotar resultados
plt.figure(figsize=(8, 6))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.title('Loss over epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Salvar modelo
torch.save(model.state_dict(), 'person_detector.pth')

In [ ]:
# Teste em algumas imagens
model.eval()
with torch.no_grad():
    for images, _ in list(val_loader)[:5]:  # visualizar 5 imagens
        image = images[0].to(DEVICE)
        predictions = model([image])[0]
        visualize_detections(image, predictions)